In [37]:
from collections import Counter
from pathlib import Path
import ast
import math

import pandas as pd

# Find project root
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data").is_dir()),
    Path.cwd(),
)

# Load already-tokenized articles
tokenized_path = project_root / "data" / "processed" / "tokenized_articles.csv"

if not tokenized_path.is_file():
    raise FileNotFoundError(f"Could not find: {tokenized_path}")

articles = pd.read_csv(tokenized_path)

if "tokens" not in articles.columns:
    raise ValueError("The CSV does not contain a 'tokens' column.")

# Convert saved token strings back into Python lists
def parse_tokens(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    return ast.literal_eval(value)

tokenized_documents = articles["tokens"].apply(parse_tokens).tolist()

# Dataset statistics
articles_processed = len(tokenized_documents)
total_tokens = sum(len(tokens) for tokens in tokenized_documents)

print(f"Loaded file: {tokenized_path}")
print(f"Articles loaded: {articles_processed}")
print(f"Total tokens: {total_tokens:,}")
print(f"Sample tokens: {tokenized_documents[0][:40]}")

Loaded file: /Users/sabirinomar/Politics-Network/notebooks/data/processed/tokenized_articles.csv
Articles loaded: 249
Total tokens: 105,769
Sample tokens: ['vigtig', 'valgkamp', 'spørgsmål', 'indlede', 'valgkamp', 'første', 'partilederdebat', 'christiansborg', 'samtlige', 'frontfigure', 'opstillingsberette', 'parti', 'sekund', 'svare', 'svare', 'socialdemokrati', 'statsminister', 'mette', 'frederiksen', 'vigtig', 'danskernes', 'tryghed', 'socialdemokrati', 'statsminister', 'mette', 'frederiksen', 'vigtig', 'danskernes', 'tryghed', 'venstre', 'jakob', 'ellemann-jensen', 'vigtigst', 'styr', 'økonomi', 'venstre', 'jakob', 'ellemann-jensen', 'vigtigst', 'styr']


In [38]:
######tf
from collections import Counter
import pandas as pd

# Count occurrences of each word in each article
document_counts = [Counter(tokens) for tokens in tokenized_documents]

# Calculate TF for every word in every article
tf_results = []

for article_id, (counts, tokens) in enumerate(
    zip(document_counts, tokenized_documents)
):
    word_count = len(tokens)

    if word_count == 0:
        continue

    for word, count in counts.items():
        tf = count / word_count

        tf_results.append({
            "article_id": article_id,
            "word": word,
            "count": count,
            "word_count": word_count,
            "tf": tf
        })

# Create DataFrame
tf_df = pd.DataFrame(tf_results)

# Sort by TF, highest first
tf_df = tf_df.sort_values(
    ["tf", "word"],
    ascending=[False, True]
).reset_index(drop=True)

# Display results
display(tf_df.head(30))

,article_id,word,count,word_count,tf
0,169,2022,50,364,0.137363
1,169,folketingsvalg,49,364,0.134615
2,210,krone,35,378,0.092593
3,125,parti,20,216,0.092593
4,133,regering,132,1621,0.081431
5,59,regering,6,76,0.078947
6,80,behandlingsgaranti,4,52,0.076923
7,242,stemme,26,340,0.076471
8,7,valg,14,203,0.068966
9,54,bornholm,25,366,0.068306


In [39]:
##idf

from collections import Counter
import math
import pandas as pd

# Total number of articles
N = len(tokenized_documents)

# Count each word only once per article
document_frequency = Counter()

for tokens in tokenized_documents:
    document_frequency.update(set(tokens))

# Calculate IDF for every unique word
idf_results = []

for word, df in document_frequency.items():
    idf = math.log(N / df)

    idf_results.append({
        "word": word,
        "document_frequency": df,
        "total_documents": N,
        "idf": idf
    })

# Create DataFrame
idf_df = pd.DataFrame(idf_results)

# Sort by IDF, highest first
idf_df = idf_df.sort_values(
    "idf",
    ascending=False
).reset_index(drop=True)

display(idf_df.head(30))

,word,document_frequency,total_documents,idf
0,ordlyd,1,249,5.517453
1,tam,1,249,5.517453
2,mylenberg,1,249,5.517453
3,13.25,1,249,5.517453
4,munch-perrin,1,249,5.517453
5,redaktionschef,1,249,5.517453
6,eventafdeling,1,249,5.517453
7,10.34,1,249,5.517453
8,dagblad,1,249,5.517453
9,11.06,1,249,5.517453


In [40]:
# Combine TF and IDF
tfidf_df = tf_df.merge(
    idf_df[["word", "idf"]],
    on="word",
    how="left"
)

# Calculate TF-IDF
tfidf_df["tfidf"] = tfidf_df["tf"] * tfidf_df["idf"]

# Sort by TF-IDF, highest first
tfidf_df = tfidf_df.sort_values(
    "tfidf",
    ascending=False
).reset_index(drop=True)

# Display results
display(tfidf_df.head(60))

,article_id,word,count,word_count,tf,idf,tfidf
0,89,symbolpolitik,6,94,0.063830,5.517453,0.352178
1,54,bornholm,25,366,0.068306,4.824306,0.329529
2,80,behandlingsgaranti,4,52,0.076923,3.725693,0.286592
3,131,tiktok,23,391,0.058824,3.908015,0.229883
4,0,vigtigst,14,269,0.052045,4.131159,0.215005
5,190,voss,29,764,0.037958,5.517453,0.209432
6,183,bornholm,9,209,0.043062,4.824306,0.207745
7,38,tiktok,13,252,0.051587,3.908015,0.201604
8,78,pension,21,317,0.066246,3.032546,0.200894
9,6,hjaltelin,11,311,0.035370,5.517453,0.195151
